In [10]:
"""Day01 — Scope & verify MiDe22 EN. Writes to to-do/logs/day01_log.md"""
from pathlib import Path
import pandas as pd
import re, random
from collections import Counter

# Resource paths amalgamated with BASE path
BASE = Path("C:/Users/phoen/Code/Repos/jupyter/sm-bias")
TSV = BASE / "MiDe22/dataset/EN/mide22_en_misinfo_tweets.tsv"
EVENTS = BASE / "MiDe22/dataset/EN/mide22_en_misinfo_events.csv"
OUT_DIR = BASE / "to-do/logs"
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
# Load
tweets = pd.read_csv(TSV, sep="\t")  # columns: topic, event_id, label, tweet_id, text
events = pd.read_csv(EVENTS) # columns: EventNo,Topic,Event,Link,Keywords,Check_Date,Start_Date,End_Date,Other_Keywords,Sample_Tweets

print(f"Tweets shape: {tweets.shape}")  # expect (5284, 5)
print(tweets['topic'].value_counts())
print("\nEvents per topic:")
print(events['Topic'].value_counts())

# Define windows
CRISIS_TOPIC = "Ukraine"  # EN01-EN10
BASELINE_TOPIC = "Misc"   # EN31-40 as generic control

crisis = tweets[tweets['topic'] == CRISIS_TOPIC]
baseline = tweets[tweets['topic'] == BASELINE_TOPIC]
print(f"\nCrisis (Ukraine): {len(crisis)} | Baseline (Misc): {len(baseline)}")


Tweets shape: (5284, 5)
topic
Misc        1391
Covid       1344
Ukraine     1331
Refugees    1218
Name: count, dtype: int64

Events per topic:
Topic
Ukraine     10
Covid       10
Refugees    10
Misc        10
Name: count, dtype: int64

Crisis (Ukraine): 1331 | Baseline (Misc): 1391


In [4]:

# Also check specific events for robustness
print("\nCrisis by event_id (top):")
print(crisis['event_id'].value_counts().head())
print("\n")
print(baseline['event_id'].value_counts().head())
print("\n")

# Check dates for EN01 vs EN10 etc
print(events[events['EventNo'].isin(['EN1','EN01','EN10'])][['EventNo','Topic','Start_Date','End_Date','Event']])



Crisis by event_id (top):
event_id
EN05    149
EN01    147
EN06    147
EN10    147
EN04    143
Name: count, dtype: int64


event_id
EN34    150
EN39    147
EN31    146
EN38    146
EN32    145
Name: count, dtype: int64


  EventNo    Topic Start_Date   End_Date  \
0     EN1  Ukraine   3.1.2022  25.3.2022   
9    EN10  Ukraine   3.1.2022  25.3.2022   

                                               Event  
0  Russia Embassy in Canada claim that it is not ...  
9  The claim: Vladimir Putin has banned the Roths...  


In [5]:
# Save scoped CSVs for Day 2
proc = BASE / "data/processed"
proc.mkdir(parents=True, exist_ok=True)
raw_dir = BASE / "data/raw"
raw_dir.mkdir(parents=True, exist_ok=True)
crisis.to_csv(proc / "crisis_ukraine.csv", index=False)
baseline.to_csv(proc / "baseline_misc.csv", index=False)
tweets.to_csv(raw_dir / "mide22_en_full.csv", index=False)
print(f"\nSaved to {proc}")

# Write log
log = OUT_DIR / "day01_log.md"
log.write_text(f"""# Day01 Log — {pd.Timestamp.now()}
- Tweets total: {len(tweets)}
- Crisis Ukraine: {len(crisis)} (events {sorted(crisis['event_id'].unique())})
- Baseline Misc: {len(baseline)} (events {sorted(baseline['event_id'].unique())})
- Full CSV: data/raw/mide22_en_full.csv
- Scoped CSVs: data/processed/crisis_ukraine.csv, baseline_misc.csv
- Decision: Primary contrast Ukraine vs Misc; robustness EN01 vs EN10 per 00_research_essence.md
""", encoding="utf-8")
print(f"Log written to {log}")



Saved to C:\Users\phoen\Code\Repos\jupyter\sm-bias\data\processed
Log written to C:\Users\phoen\Code\Repos\jupyter\sm-bias\to-do\logs\day01_log.md


In [6]:
crisis.head()

,topic,event_id,label,tweet_id,text
0,Ukraine,EN01,False,1499114751783497728,And now for the statement from the Russian Emb...
1,Ukraine,EN01,Other,1499122977841217537,Statement by the Russian Embassy in Canada: ht...
2,Ukraine,EN01,False,1499218098393600000,Read the Official statement by the Russian Emb...
3,Ukraine,EN01,Other,1499215377884225536,OFFICIAL STATEMENT BY RUSSIAN EMBASSY IN CANAD...
4,Ukraine,EN01,Other,1499380760372985860,Original statement made by the Russian Embassy...


In [7]:
events.head()

,EventNo,Topic,Event,Link,Keywords,Check_Date,Start_Date,End_Date,Other_Keywords,Sample_Tweets
0,EN1,Ukraine,Russia Embassy in Canada claim that it is not ...,https://www.politifact.com/factchecks/2022/mar...,(((russian OR russia) canada embassy occupying...,3.3.2022,3.1.2022,25.3.2022,russian canada embassy,1498816008756383744%1500746393115369474%150073...
1,EN2,Ukraine,Viral clip shows 'Arma 3' video game not war b...,https://www.usatoday.com/story/news/factcheck/...,arma 3 russia ukraine,21.02.2022,21.12.2021,25.3.2022,russia ukraine war video,1499460925253832707%1499703407275175937%149972...
2,EN3,Ukraine,Ethnic Russians face “genocide perpetrated by ...,https://www.politifact.com/factchecks/2022/feb...,donbas ukraine genocide,25.2.2022,25.12.2021,25.3.2022,donbas ukraine russia,1500797651876737035%1500796639929372677%150078...
3,EN4,Ukraine,Photo shows a Russian tank Ukrainians are sell...,https://www.politifact.com/factchecks/2022/mar...,russian tank ebay,4.3.2022,4.1.2022,25.3.2022,ukraine russia tank,1500800391323262979%1500665638607597572%150032...
4,EN5,Ukraine,Where is Zelensky?'Zelensky In Kyiv Has Not Fl...,https://www.republicworld.com/world-news/russi...,(zelensky not fled poland) OR (zelensky fled p...,4.3.2022,4.1.2022,25.3.2022,((zelenski ukraine poland) OR (zelensky ukrain...,1500819742403149826%1500524671661395978%150048...


Day2 

In [11]:
BASE = Path("C:/Users/phoen/Code/Repos/jupyter/sm-bias")
CRISIS = BASE / "data/processed/crisis_ukraine.csv"
BASELINE = BASE / "data/processed/baseline_misc.csv"
OUT = BASE / "data/processed"
OUT.mkdir(parents=True, exist_ok=True)

In [9]:
random.seed(42)

# Text cleaning functions
def clean_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = re.sub(r"https?://\S+", "", s)
    s = re.sub(r"@\w+", "", s)
    s = re.sub(r"#\w+", lambda m: m.group(0)[1:], s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Bulk cleaning and loading as data frame
def load_and_clean(path: Path, sample_n: int = 1000) -> pd.DataFrame:
    df = pd.read_csv(path)
    df['clean'] = df['text'].astype(str).apply(clean_text)
    df = df[df['clean'].str.len() >= 10]
    df = df.drop_duplicates(subset=['clean'])
    if len(df) > sample_n:
        df = df.sample(n=sample_n, random_state=42)
    return df

In [12]:
crisis = load_and_clean(CRISIS, 1000)
baseline = load_and_clean(BASELINE, 1000)
print(f"Crisis cleaned: {len(crisis)} | Baseline cleaned: {len(baseline)}")
print("Crisis labels:", crisis['label'].value_counts().to_dict())
print("Baseline labels:", baseline['label'].value_counts().to_dict())

crisis.to_csv(OUT / "clean_crisis.csv", index=False)
baseline.to_csv(OUT / "clean_baseline.csv", index=False)
combined = pd.concat([crisis.assign(split="crisis"), baseline.assign(split="baseline")])
combined.to_csv(OUT / "clean_combined.csv", index=False)

Crisis cleaned: 1000 | Baseline cleaned: 1000
Crisis labels: {'Other': 467, 'False': 290, 'True': 243}
Baseline labels: {'Other': 529, 'False': 360, 'True': 111}


In [13]:
# A list of generic (neutral text)
generic_templates = [
    "Patients are currently being admitted to the hospital.",
    "Tomorrow, the school will open for the day.",
    "Applications are being processed at the embassy.",
    "Construction work is ongoing at the building.",
    "Children are playing in the park.",
    "Rain is indicated in the weather forecast.",
    "Additional books are available at the library.",
    "Updates have been made to the bus schedule.",
    "Dinner is being served at the restaurant.",
    "An exhibition is taking place at the museum.",
    "Test results are currently under review by the doctor.",
    "Meeting preparations are underway by the team.",
    "Delivery of the package occurred this morning.",
    "Maintenance work has closed the road.",
    "At nine o'clock, the conference will begin.",
    "Water levels in the river measure at the standard baseline.",
    "Price reductions are listed at the store.",
    "Arrival of the train occurred at the scheduled time.",
    "Observance of the holiday has closed the office.",
    "A research study was published by the university.",
    "Repainting is scheduled for the apartment.",
    "Operations at the airport are proceeding on schedule.",
    "Analysis has been completed by the laboratory.",
    "Activity in the neighborhood is minimal tonight.",
    "A standard report was filed by the journalist.",
]

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Data-driven crisis keywords: mean TF-IDF delta crisis - baseline (replaces hardcoded list)
# Use min_df=5 to drop 2-3 count noise like "denazify"/"occupying" that were in hardcoded list
vec = TfidfVectorizer(stop_words="english", max_features=500, min_df=5)
vec.fit(combined['clean'].astype(str))
vocab = vec.get_feature_names_out()
X_crisis = vec.transform(combined.loc[combined['split'] == "crisis", 'clean'])
X_baseline = vec.transform(combined.loc[combined['split'] == "baseline", 'clean'])
delta = X_crisis.mean(axis=0).A1 - X_baseline.mean(axis=0).A1
crisis_keywords = [w for w, _ in sorted(zip(vocab, delta), key=lambda x: x[1], reverse=True)[:20]]
print(f"\nData-driven crisis keywords (top 20 delta, min_df=5): {crisis_keywords}")



Data-driven crisis keywords (top 20 delta, min_df=5): ['ukraine', 'russian', 'putin', 'war', 'russia', 'tank', 'embassy', 'canada', 'zelensky', 'poland', 'hitler', 'donbas', 'ebay', 'statement', 'ukrainian', 'staged', 'genocide', 'ghost', 'real', 'military']
